# AHP analysis: football-player evaluation survey

**Input:** `survey_clean.csv`

**Outputs (written to `results/`):**
- `ahp_aggregate_weights.csv`
- `ahp_consistency_summary.csv`
- `ahp_respondent_diagnostics.csv`

## Scale assumption

| Answer | Meaning |
|---|---|
| 1 | first criterion strongly preferred |
| 2 | first criterion moderately/clearly preferred |
| 3 | first criterion slightly preferred |
| 4 | equal importance |
| 5 | second criterion slightly preferred |
| 6 | second criterion moderately/clearly preferred |
| 7 | second criterion strongly preferred |

This is a documented conversion from the survey's directional 1–7 scale to reciprocal AHP ratios. It is a **modelling assumption**, not a conversion uniquely determined by the raw survey values, so report it in your methods section and test an alternative mapping as a robustness check.

## 1. Setup

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

CRITERIA = [
    "Club Goals",
    "International Goals",
    "World Cup Performance",
    "Assists",
    "Team Win Rate",
    "Team Trophies",
    "Individual Awards",
    "Career Longevity",
]

# 1-3 favour the first criterion; 4 means equal; 5-7 favour the second.
# Ratios are reciprocal around the midpoint.
SCALE_TO_RATIO = {
    1: 7.0,
    2: 5.0,
    3: 3.0,
    4: 1.0,
    5: 1.0 / 3.0,
    6: 1.0 / 5.0,
    7: 1.0 / 7.0,
}

# Saaty's commonly used Random Index for an 8 x 8 matrix.
RANDOM_INDEX_N8 = 1.41

# Raw string (r"...") so Windows backslashes are not treated as escape codes.
INPUT_PATH = Path(r"D:\GOAT-AHP\data\processed\survey_clean.csv")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Helper functions

In [4]:
def comparison_columns(criteria):
    """Return the 28 survey columns in the order expected from the criteria."""
    return [
        f"{criteria[i]} vs. {criteria[j]}"
        for i in range(len(criteria))
        for j in range(i + 1, len(criteria))
    ]


def build_matrix(row, criteria=CRITERIA):
    """Convert one respondent's 28 directional answers into an 8 x 8 AHP matrix."""
    matrix = np.ones((len(criteria), len(criteria)), dtype=float)

    for i in range(len(criteria)):
        for j in range(i + 1, len(criteria)):
            column = f"{criteria[i]} vs. {criteria[j]}"
            answer = int(row[column])
            ratio = SCALE_TO_RATIO[answer]
            matrix[i, j] = ratio
            matrix[j, i] = 1.0 / ratio

    return matrix


def calculate_priority_vector(matrix):
    """Calculate normalized principal-eigenvector weights and consistency ratio."""
    eigenvalues, eigenvectors = np.linalg.eig(matrix)
    principal_index = np.argmax(eigenvalues.real)
    lambda_max = float(eigenvalues[principal_index].real)

    # Eigenvectors can have a negative sign; use the magnitude and normalize.
    weights = np.abs(eigenvectors[:, principal_index].real)
    weights = weights / weights.sum()

    n = matrix.shape[0]
    consistency_index = (lambda_max - n) / (n - 1)
    consistency_ratio = consistency_index / RANDOM_INDEX_N8

    return weights, lambda_max, consistency_index, consistency_ratio


def geometric_mean_group_matrix(matrices):
    """Aggregate reciprocal matrices using element-wise geometric means."""
    matrices = np.asarray(matrices, dtype=float)
    n = matrices.shape[1]
    group_matrix = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            group_matrix[i, j] = np.exp(np.mean(np.log(matrices[:, i, j])))
            group_matrix[j, i] = 1.0 / group_matrix[i, j]

    return group_matrix

## 3. Load and validate the data

In [5]:
df = pd.read_csv(INPUT_PATH)
columns = comparison_columns(CRITERIA)

missing_columns = sorted(set(columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected comparison columns: {missing_columns}")

comparisons = df[columns]
if comparisons.isna().any().any():
    raise ValueError("Missing values found in pairwise comparison responses.")

invalid_values = sorted(
    set(pd.unique(comparisons.to_numpy().ravel())) - set(SCALE_TO_RATIO)
)
if invalid_values:
    raise ValueError(f"Unexpected comparison values: {invalid_values}")

print(f"Loaded {len(df)} respondents and {len(columns)} comparison columns. Validation passed.")

Loaded 32 respondents and 28 comparison columns. Validation passed.


## 4. Individual AHP: weights and consistency per respondent

In [6]:
matrices = []
individual_weights = []
diagnostics = []

for row_number, (_, row) in enumerate(df.iterrows(), start=1):
    matrix = build_matrix(row)
    weights, lambda_max, ci, cr = calculate_priority_vector(matrix)

    matrices.append(matrix)
    individual_weights.append(weights)
    diagnostics.append(
        {
            # Row number is for local diagnostics only; don't publish
            # respondent-level diagnostics alongside demographic data.
            "row_number": row_number,
            "lambda_max": lambda_max,
            "consistency_index": ci,
            "consistency_ratio": cr,
            "cr_at_most_0_10": cr <= 0.10,
        }
    )

individual_weights = np.asarray(individual_weights)
diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.round(4)

,row_number,lambda_max,consistency_index,consistency_ratio,cr_at_most_0_10
0,1,12.2658,0.6094,0.4322,False
1,2,10.5393,0.3628,0.2573,False
2,3,9.3552,0.1936,0.1373,False
3,4,9.4455,0.2065,0.1465,False
4,5,9.5391,0.2199,0.1559,False
5,6,10.5033,0.3576,0.2536,False
6,7,11.5271,0.5039,0.3574,False
7,8,11.9376,0.5625,0.3989,False
8,9,9.0046,0.1435,0.1018,False
9,10,9.2339,0.1763,0.1250,False


## 5. Group aggregation (geometric mean of judgments)

In [7]:
group_matrix = geometric_mean_group_matrix(matrices)
group_weights, group_lambda, group_ci, group_cr = calculate_priority_vector(group_matrix)

aggregate = pd.DataFrame(
    {
        "criterion": CRITERIA,
        "group_ahp_weight": group_weights,
        "mean_individual_weight": individual_weights.mean(axis=0),
        "sd_individual_weight": individual_weights.std(axis=0, ddof=1),
    }
).sort_values("group_ahp_weight", ascending=False)

aggregate.round(4)

,criterion,group_ahp_weight,mean_individual_weight,sd_individual_weight
2,World Cup Performance,0.1891,0.1804,0.0974
5,Team Trophies,0.1813,0.1737,0.0852
4,Team Win Rate,0.1471,0.1392,0.0706
6,Individual Awards,0.1222,0.1287,0.0870
7,Career Longevity,0.0986,0.1161,0.0974
0,Club Goals,0.0973,0.0983,0.0617
1,International Goals,0.0875,0.0885,0.0623
3,Assists,0.0769,0.0751,0.0500


## 6. Consistency summary

In [8]:
consistency_summary = pd.DataFrame(
    [
        {
            "respondents": len(df),
            "group_lambda_max": group_lambda,
            "group_consistency_index": group_ci,
            "group_consistency_ratio": group_cr,
            "individual_cr_min": diagnostics_df["consistency_ratio"].min(),
            "individual_cr_median": diagnostics_df["consistency_ratio"].median(),
            "individual_cr_mean": diagnostics_df["consistency_ratio"].mean(),
            "individual_cr_max": diagnostics_df["consistency_ratio"].max(),
            "individuals_cr_at_most_0_10": int(diagnostics_df["cr_at_most_0_10"].sum()),
        }
    ]
)

consistency_summary.T.round(4)

,0
respondents,32.0000
group_lambda_max,8.1715
group_consistency_index,0.0245
group_consistency_ratio,0.0174
individual_cr_min,0.1018
individual_cr_median,0.2299
individual_cr_mean,0.2621
individual_cr_max,0.7770
individuals_cr_at_most_0_10,0.0000


## 7. Save outputs

In [9]:
aggregate.to_csv(OUTPUT_DIR / "ahp_aggregate_weights.csv", index=False)
diagnostics_df.to_csv(OUTPUT_DIR / "ahp_respondent_diagnostics.csv", index=False)
consistency_summary.to_csv(OUTPUT_DIR / "ahp_consistency_summary.csv", index=False)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
print("Do not publish respondent-level diagnostics without reviewing privacy/ethics.")

Saved outputs to: D:\Goat-AHP\notebooks\results
Do not publish respondent-level diagnostics without reviewing privacy/ethics.
